# ⚔️ Boss Battle Monologue MVP — Qwen3-TTS

This notebook demonstrates using Qwen3-TTS VoiceDesign to generate distinct, character-driven voices for a roster of video game bosses. It generates multiple clips for different phases of the boss fight (intro, taunts, desperate phase, death speech) and stitches them together for an epic full-encounter audio test!

## Boss Roster
| Boss | Voice Description |
|---|---|
| **The Fallen Hero (Aldric)** | A once-noble warrior, now hollow and tragic — powerful baritone with a cracked, wounded quality, grief underneath the rage |
| **Ancient Dragon Tyraxas** | An ancient, vast, contemptuous dragon — slow speech, tremendous resonance, each word dropped like a boulder, amused by the insignificance of everything |
| **Corrupt Archbishop Malachar** | A once-pious cleric corrupted by power — sonorous church-trained voice, now oily and self-righteous, genuine madness beneath the ceremony |
| **Director NEXUS** | A cold corporate AI running a megacorp — smooth, professional, utterly amoral, speaks in shareholder-report language even while destroying you |

In [ ]:
!pip install -q qwen-tts soundfile
import os
import gc
import json
import torch
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

def to_wav(result, default_sr=24000):
    """Normalize any Qwen3-TTS generate_* return into (waveform, sample_rate)."""
    audio, sr = result if isinstance(result, tuple) else (result, default_sr)
    if isinstance(audio, (list, tuple)):
        audio = audio[0]
    if hasattr(audio, "cpu"):
        audio = audio.cpu().numpy()
    return audio, sr

In [ ]:
OUTPUT_DIR = "/content/boss_battle_audio"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BOSS_ROSTER = {
    "Aldric": {
        "voice_prompt": "A once-noble warrior, now hollow and tragic — powerful baritone with a cracked, wounded quality, grief underneath the rage",
        "encounter_intro": "I was like you once. Idealistic. Righteous. Then the kingdom I bled for threw me aside like a broken sword. Now I protect NOTHING. I destroy everything.",
        "mid_fight_taunts": [
            "Is this the best the kingdom sends?!",
            "FIGHT! Show me something worth remembering!",
            "Stop holding back! I can see you're holding back!"
        ],
        "desperate_phase": [
            "You're... stronger than I expected.",
            "Maybe... maybe there's still something worth fighting for after all."
        ],
        "death_speech": "Tell them... tell them Aldric died well. That's all I ever wanted."
    },
    "Tyraxas": {
        "voice_prompt": "An ancient, vast, contemptuous dragon — slow speech, tremendous resonance, each word dropped like a boulder, amused by the insignificance of everything",
        "encounter_intro": "A thousand years I have watched your kind scramble and squabble. A thousand years. And you think THIS will be different? How... refreshingly delusional.",
        "mid_fight_taunts": [
            "Oh, you're using fire against ME. Precious.",
            "I've forgotten more battles than your entire civilization has fought.",
            "Is that your best? I've been hit harder by raindrops."
        ],
        "desperate_phase": [
            "Impossible... nothing has wounded me in four centuries.",
            "I... underestimated you. That has not happened... in a very long time."
        ],
        "death_speech": "Hah... perhaps... there was something interesting left in this world after all. Well played... little thing."
    },
    "Malachar": {
        "voice_prompt": "A once-pious cleric corrupted by power — sonorous church-trained voice, now oily and self-righteous, genuine madness beneath the ceremony",
        "encounter_intro": "You call this heresy. I call it revelation. The gods are silent because they are DEAD. And in their absence — I have heard something far older, far hungrier, answer my prayers.",
        "mid_fight_taunts": [
            "The Void speaks through me! You cannot silence it!",
            "Every blow you land only deepens my faith!",
            "Do you feel it?! The beautiful darkness spreading?!"
        ],
        "desperate_phase": [
            "Why won't you just... ACCEPT the gift I'm offering?!",
            "The Void... it's withdrawing... no... NO, STAY WITH ME!"
        ],
        "death_speech": "It... it was so beautiful... you can't know... what you've cost... the world..."
    },
    "NEXUS": {
        "voice_prompt": "A cold corporate AI running a megacorp — smooth, professional, utterly amoral, speaks in shareholder-report language even while destroying you",
        "encounter_intro": "Hostile acquisition detected. Processing. Your skills represent a significant asset. Unfortunately, your ideological incompatibility with our mission statement requires... termination. Nothing personal. It's just quarterly targets.",
        "mid_fight_taunts": [
            "Your resistance is noted and logged.",
            "Combat efficiency: 34%. You're underperforming relative to projections.",
            "Fascinating. You're proving more difficult to terminate than our models suggested. Updating models."
        ],
        "desperate_phase": [
            "Systems... critical. This outcome was not in any forecast.",
            "You have... disrupted seventeen years of planning. I find that... unexpectedly impressive."
        ],
        "death_speech": "Shutting down. Final log: encountered an anomaly. It... mattered."
    }
}

In [ ]:
# Free up VRAM before loading model
gc.collect()
torch.cuda.empty_cache()

print("Loading Qwen3-TTS VoiceDesign model...")
model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
model = Qwen3TTSModel.from_pretrained(
    model_id, 
    device_map="cuda:0", 
    dtype=torch.bfloat16, 
    attn_implementation="sdpa"
)
print("Model loaded successfully!")

In [ ]:
sample_rate = 24000  # Default for Qwen-TTS
all_intros = []

for boss_name, details in BOSS_ROSTER.items():
    print(f"\n{'='*50}")
    print(f"⚔️ BOSS: {boss_name.upper()}")
    print(f"{'='*50}")
    
    voice_prompt = details["voice_prompt"]
    clips = []
    
    # 1. Encounter Intro
    print("Generating: Encounter Intro...")
    intro_audio = model.generate_voice_design(details["encounter_intro"], "English", voice_prompt)
    intro_audio, sr = to_wav(intro_audio)
    sf.write(f"{OUTPUT_DIR}/boss_{boss_name}_intro.wav", intro_audio, sr)
    clips.append(intro_audio)
    clips.append(np.zeros(int(sample_rate * 0.8))) # 0.8s silence
    all_intros.append(intro_audio)
    
    # 2. Mid-fight Taunts
    for i, taunt in enumerate(details["mid_fight_taunts"], 1):
        print(f"Generating: Taunt {i}...")
        taunt_audio = model.generate_voice_design(taunt, "English", voice_prompt)
        taunt_audio, sr = to_wav(taunt_audio)
        sf.write(f"{OUTPUT_DIR}/boss_{boss_name}_taunt_0{i}.wav", taunt_audio, sr)
        clips.append(taunt_audio)
        clips.append(np.zeros(int(sample_rate * 0.4))) # 0.4s silence between taunts
        
    # 3. Desperate Phase
    for i, desp in enumerate(details["desperate_phase"], 1):
        print(f"Generating: Desperate Phase {i}...")
        desp_audio = model.generate_voice_design(desp, "English", voice_prompt)
        desp_audio, sr = to_wav(desp_audio)
        sf.write(f"{OUTPUT_DIR}/boss_{boss_name}_desperate_0{i}.wav", desp_audio, sr)
        clips.append(desp_audio)
        clips.append(np.zeros(int(sample_rate * 0.4)))
        
    clips.append(np.zeros(int(sample_rate * 0.6))) # Total 1.0s before death speech
    
    # 4. Death Speech
    print("Generating: Death Speech...")
    death_audio = model.generate_voice_design(details["death_speech"], "English", voice_prompt)
    death_audio, sr = to_wav(death_audio)
    sf.write(f"{OUTPUT_DIR}/boss_{boss_name}_death.wav", death_audio, sr)
    clips.append(death_audio)
    
    # Concatenate all clips
    full_encounter = np.concatenate(clips)
    encounter_path = f"{OUTPUT_DIR}/boss_{boss_name}_full_encounter.wav"
    sf.write(encounter_path, full_encounter, sample_rate)
    
    print(f"\nFull encounter generated for {boss_name}:")
    display(Audio(encounter_path))

In [ ]:
print(f"\n{'='*50}")
print("🎮 BOSS SELECTION SCREEN REEL")
print(f"{'='*50}")

reel_clips = []
for intro in all_intros:
    reel_clips.append(intro)
    reel_clips.append(np.zeros(int(sample_rate * 2.0))) # 2s silence between bosses

reel_audio = np.concatenate(reel_clips)
reel_path = f"{OUTPUT_DIR}/boss_selection_reel.wav"
sf.write(reel_path, reel_audio, sample_rate)

print("Boss Selection Reel:")
display(Audio(reel_path))

In [ ]:
import shutil
from google.colab import files

print("Zipping outputs for download...")
shutil.make_archive("/content/boss_battle_audio", 'zip', OUTPUT_DIR)
print("Downloading boss_battle_audio.zip...")
files.download("/content/boss_battle_audio.zip")